# GVH Diagonal Cubic 0.2.23.1 — Weak-Field Coefficient Extraction for PPN

**Auteur : Charlemagne O Laurince**

---

## Objectif

Ce notebook extrait, sans les inventer, les coefficients faibles nécessaires au raccordement PPN :

\[
g_{tt}^{\rm GVH}
=
-1+2a_1u-2a_2u^2+\mathcal O(u^3),
\]

\[
g_{ij}^{\rm GVH}
=
\left(
1+2b_1u+b_2u^2+\mathcal O(u^3)
\right)\delta_{ij},
\]

avec :

\[
u=\frac{G_NM}{rc^2}.
\]

Le matching établi dans `0.2.22` est :

\[
\boxed{
\gamma_{\rm GVH}=\frac{b_1}{a_1}
}
\]

et :

\[
\boxed{
\beta_{\rm GVH}=\frac{a_2}{a_1^2}
}.
\]

---

## Sources théoriques

Le notebook utilise les résultats de :

```text
0.2.21 — Equivalence Principle PPN Constraints
0.2.22 — Static Spherical Solution PPN Derivation
0.2.23 — Covariant Static Field Equations Source Matching
```

Il distingue deux branches :

### Branche A — source parfaite isotrope avec couplage strict à la partie sans trace

Dans cette branche :

\[
\Pi_{\mu\nu}[T]=0,
\qquad
Q_D=0,
\qquad
d(r)=0.
\]

L’extérieur est Schwarzschild et :

\[
a_1=1,\qquad
a_2=1,\qquad
b_1=1.
\]

Cette conclusion est dérivée pour cette branche précise.

### Branche B — charge directionnelle non nulle

Une charge \(Q_D\neq0\) exige au moins :

- anisotropie de pression ;
- rotation ;
- marées ;
- couplage effectif à la densité ;
- mélange avec la courbure ;
- composantes temporelles supplémentaires.

Dans cette branche, `0.2.23` indique encore comme non dérivés :

\[
\Pi_{\mu\nu}[T],
\qquad
\alpha_t,
\qquad
\alpha_s,
\]

ainsi que les coefficients non linéaires contrôlant \(\beta\).

---

## Principe de sûreté scientifique

Le notebook ne doit pas transformer la branche Schwarzschild limitée en prédiction universelle de tout GVH.

Statuts possibles :

```text
PASS-STRICT-ISOTROPIC-GR-BRANCH
BLOCKED-NONTRIVIAL-GVH-COEFFICIENTS-NOT-DERIVED
PASS-GLOBAL-GVH-WEAK-FIELD-COEFFICIENTS
BLOCKED-SOURCE-NOTEBOOKS-NOT-FOUND
```

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

print("Python :", sys.version)
print("SymPy :", sp.__version__)

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
SymPy : 1.14.0


# 1. Dépôt et chemins

In [2]:
REPOSITORY_URL = "https://github.com/col38470682/Univers.git"
REPOSITORY_DIR = Path("/content/Univers")
PROJECT_ROOT = REPOSITORY_DIR / "gvh_diagonal_cubic"

if (REPOSITORY_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)

PROCESSED_PPN_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ppn"
)

EXPORT_DIR = PROJECT_ROOT / "exports"

for directory in [
    PROCESSED_PPN_DIR,
    EXPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("PROJECT_ROOT :", PROJECT_ROOT)
print("PROCESSED_PPN_DIR :", PROCESSED_PPN_DIR)

PROJECT_ROOT : /content/Univers/gvh_diagonal_cubic
PROCESSED_PPN_DIR : /content/Univers/gvh_diagonal_cubic/data/processed/ppn


# 2. Découverte obligatoire de 0.2.21, 0.2.22 et 0.2.23

In [3]:
NOTEBOOK_PATTERNS = {
    "0.2.21": "*0.2.21*Equivalence*Principle*PPN*Constraints*.ipynb",
    "0.2.22": "*0.2.22*Static*Spherical*PPN*Derivation*.ipynb",
    "0.2.23": "*0.2.23*Covariant*Static*Field*Source*Matching*.ipynb",
}

resolved_notebooks = {}

for notebook_id, pattern in NOTEBOOK_PATTERNS.items():
    matches = sorted(
        PROJECT_ROOT.rglob(pattern)
    )

    resolved_notebooks[
        notebook_id
    ] = (
        matches[0]
        if matches
        else None
    )

source_notebook_df = pd.DataFrame([
    {
        "notebook_id": notebook_id,
        "resolved_path": (
            str(path)
            if path is not None
            else ""
        ),
        "found": path is not None,
    }
    for notebook_id, path
    in resolved_notebooks.items()
])

source_notebook_df

,notebook_id,resolved_path,found
0,0.2.21,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb,True
1,0.2.22,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb,True
2,0.2.23,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2B_astrophysics_validation/GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb,True


In [4]:
all_source_notebooks_found = bool(
    source_notebook_df[
        "found"
    ].all()
)

if not all_source_notebooks_found:
    print(
        "ATTENTION : un notebook source est absent. "
        "L’extraction restera bloquée."
    )

# 2A. Interface PPN héritée de 0.2.21

`0.2.21` fixe l'interface phénoménologique utilisée par la suite :

\[
\gamma_{\rm eff}=1+\epsilon_D\gamma_D,
\qquad
\beta_{\rm eff}=1+\epsilon_D\beta_D.
\]

Il ne dérive pas encore \(\gamma_D\) et \(\beta_D\) depuis les équations statiques.  
Cette dérivation doit donc provenir de `0.2.22` et `0.2.23`.

Cette cellule empêche de confondre **paramétrisation PPN** et **prédiction dynamique GVH**.

In [5]:
ppn_interface_021_df = pd.DataFrame([
    {
        "quantity": "gamma_eff",
        "expression": "1 + epsilon_D * gamma_D",
        "role": "phenomenological interface",
        "derived_from_static_GVH": False,
        "source": "0.2.21",
    },
    {
        "quantity": "beta_eff",
        "expression": "1 + epsilon_D * beta_D",
        "role": "phenomenological interface",
        "derived_from_static_GVH": False,
        "source": "0.2.21",
    },
])

ppn_interface_021_df

,quantity,expression,role,derived_from_static_GVH,source
0,gamma_eff,1 + epsilon_D * gamma_D,phenomenological interface,False,0.2.21
1,beta_eff,1 + epsilon_D * beta_D,phenomenological interface,False,0.2.21


# 3. Résultats théoriques établis par 0.2.22

In [6]:
a1, a2, b1 = sp.symbols(
    "a1 a2 b1",
    real=True,
)

gamma_GVH = sp.simplify(
    b1 / a1
)

beta_GVH = sp.simplify(
    a2 / a1**2
)

matching_df = pd.DataFrame([
    {
        "quantity": "gamma_GVH",
        "expression": str(
            gamma_GVH
        ),
        "status": (
            "DERIVED FOR PARAMETRIC STATIC FAMILY"
        ),
        "source": "0.2.22",
    },
    {
        "quantity": "beta_GVH",
        "expression": str(
            beta_GVH
        ),
        "status": (
            "DERIVED FOR PARAMETRIC STATIC FAMILY"
        ),
        "source": "0.2.22",
    },
])

matching_df

,quantity,expression,status,source
0,gamma_GVH,b1/a1,DERIVED FOR PARAMETRIC STATIC FAMILY,0.2.22
1,beta_GVH,a2/a1**2,DERIVED FOR PARAMETRIC STATIC FAMILY,0.2.22


À premier ordre dans la paramétrisation :

\[
a_1=1+\epsilon_D\alpha_1,
\]

\[
a_2=1+\epsilon_D\alpha_2,
\]

\[
b_1=1+\epsilon_D\sigma_1,
\]

on obtient :

\[
\gamma_{\rm GVH}-1
=
\epsilon_D
(\sigma_1-\alpha_1)
+\mathcal O(\epsilon_D^2),
\]

\[
\beta_{\rm GVH}-1
=
\epsilon_D
(\alpha_2-2\alpha_1)
+\mathcal O(\epsilon_D^2).
\]

In [7]:
epsilon_D = sp.symbols(
    "epsilon_D",
    real=True,
)

alpha1, alpha2, sigma1 = sp.symbols(
    "alpha1 alpha2 sigma1",
    real=True,
)

parameter_substitutions = {
    a1: 1 + epsilon_D * alpha1,
    a2: 1 + epsilon_D * alpha2,
    b1: 1 + epsilon_D * sigma1,
}

delta_gamma_linear = sp.series(
    gamma_GVH.subs(
        parameter_substitutions
    ) - 1,
    epsilon_D,
    0,
    2,
).removeO()

delta_beta_linear = sp.series(
    beta_GVH.subs(
        parameter_substitutions
    ) - 1,
    epsilon_D,
    0,
    2,
).removeO()

linear_mapping_df = pd.DataFrame([
    {
        "quantity": "gamma_GVH - 1",
        "expression": str(
            sp.expand(
                delta_gamma_linear
            )
        ),
    },
    {
        "quantity": "beta_GVH - 1",
        "expression": str(
            sp.expand(
                delta_beta_linear
            )
        ),
    },
])

linear_mapping_df

,quantity,expression
0,gamma_GVH - 1,-alpha1*epsilon_D + epsilon_D*sigma1
1,beta_GVH - 1,-2*alpha1*epsilon_D + alpha2*epsilon_D


# 4. Résultats établis par 0.2.23

In [8]:
source_matching_results_df = pd.DataFrame([
    {
        "quantity": "radial field equation",
        "result": (
            "d'' + 2 d'/r - mu_D^2 d = -kappa_D rho"
        ),
        "status": (
            "DERIVED FROM CHOSEN REDUCED PROTOTYPE"
        ),
    },
    {
        "quantity": "exterior solution",
        "result": (
            "d_out = Q_D exp(-mu_D r)/r"
        ),
        "status": "DERIVED",
    },
    {
        "quantity": "perfect isotropic fluid",
        "result": (
            "strict traceless source gives Pi_mn[T] = 0"
        ),
        "status": "DERIVED CONDITIONALLY",
    },
    {
        "quantity": "strict isotropic exterior",
        "result": (
            "Q_D = 0 and Schwarzschild exterior"
        ),
        "status": "DERIVED CONDITIONALLY",
    },
    {
        "quantity": "alpha_t and alpha_s",
        "result": "metric backreaction coefficients",
        "status": "NOT DERIVED",
    },
    {
        "quantity": "nonlinear beta coefficients",
        "result": "required for beta_GVH",
        "status": "NOT DERIVED",
    },
    {
        "quantity": "unique GVH PPN prediction",
        "result": "global model prediction",
        "status": "NOT AVAILABLE",
    },
])

source_matching_results_df

,quantity,result,status
0,radial field equation,d'' + 2 d'/r - mu_D^2 d = -kappa_D rho,DERIVED FROM CHOSEN REDUCED PROTOTYPE
1,exterior solution,d_out = Q_D exp(-mu_D r)/r,DERIVED
2,perfect isotropic fluid,strict traceless source gives Pi_mn[T] = 0,DERIVED CONDITIONALLY
3,strict isotropic exterior,Q_D = 0 and Schwarzschild exterior,DERIVED CONDITIONALLY
4,alpha_t and alpha_s,metric backreaction coefficients,NOT DERIVED
5,nonlinear beta coefficients,required for beta_GVH,NOT DERIVED
6,unique GVH PPN prediction,global model prediction,NOT AVAILABLE


# 5. Branche stricte isotrope

In [9]:
STRICT_ISOTROPIC_BRANCH = {
    "source_type": (
        "perfect isotropic fluid"
    ),
    "source_projector": (
        "strict spatial traceless projection"
    ),
    "Pi_mn_T": "0",
    "Q_D": "0",
    "directional_field_exterior": "0",
    "exterior_metric": "Schwarzschild",
}

strict_a1 = sp.Integer(1)
strict_a2 = sp.Integer(1)
strict_b1 = sp.Integer(1)

strict_gamma = sp.simplify(
    strict_b1 / strict_a1
)

strict_beta = sp.simplify(
    strict_a2 / strict_a1**2
)

strict_branch_df = pd.DataFrame([{
    "branch": "strict_isotropic_traceless_source",
    "a1": str(strict_a1),
    "a2": str(strict_a2),
    "b1": str(strict_b1),
    "gamma_GVH": str(strict_gamma),
    "beta_GVH": str(strict_beta),
    "scope": (
        "Perfect isotropic source with strict "
        "traceless-stress coupling"
    ),
    "global_GVH_prediction": False,
}])

strict_branch_df

,branch,a1,a2,b1,gamma_GVH,beta_GVH,scope,global_GVH_prediction
0,strict_isotropic_traceless_source,1,1,1,1,1,Perfect isotropic source with strict traceless-stress coupling,False


# 6. Vérification de la métrique faible de la branche stricte

In [10]:
u = sp.symbols(
    "u",
    real=True,
)

gtt_strict = sp.expand(
    -1
    + 2 * strict_a1 * u
    - 2 * strict_a2 * u**2
)

gspace_strict = sp.expand(
    1
    + 2 * strict_b1 * u
    + sp.Rational(3, 2) * u**2
)

expected_gtt_GR = (
    -1
    + 2 * u
    - 2 * u**2
)

expected_gspace_GR = (
    1
    + 2 * u
    + sp.Rational(3, 2) * u**2
)

strict_metric_pass = bool(
    sp.simplify(
        gtt_strict
        - expected_gtt_GR
    ) == 0
    and
    sp.simplify(
        gspace_strict
        - expected_gspace_GR
    ) == 0
)

strict_ppn_pass = bool(
    strict_gamma == 1
    and strict_beta == 1
)

strict_validation_df = pd.DataFrame([
    {
        "test": "Schwarzschild weak g_tt",
        "pass": strict_metric_pass,
    },
    {
        "test": "strict gamma = 1",
        "pass": strict_gamma == 1,
    },
    {
        "test": "strict beta = 1",
        "pass": strict_beta == 1,
    },
])

strict_validation_df

,test,pass
0,Schwarzschild weak g_tt,True
1,strict gamma = 1,True
2,strict beta = 1,True


# 7. Branche directionnelle non triviale

In [11]:
nontrivial_requirements_df = pd.DataFrame([
    {
        "required_quantity": "Pi_mn[T]",
        "purpose": (
            "derive the physical source of Q_D"
        ),
        "status": "MISSING",
    },
    {
        "required_quantity": "alpha_t",
        "purpose": (
            "derive temporal metric backreaction"
        ),
        "status": "MISSING",
    },
    {
        "required_quantity": "alpha_s",
        "purpose": (
            "derive spatial metric backreaction"
        ),
        "status": "MISSING",
    },
    {
        "required_quantity": (
            "second-order nonlinear coefficients"
        ),
        "purpose": (
            "derive a2 and beta_GVH"
        ),
        "status": "MISSING",
    },
    {
        "required_quantity": (
            "isotropic-coordinate transformation"
        ),
        "purpose": (
            "separate gauge coefficients from PPN coefficients"
        ),
        "status": "REQUIRED_FOR_FINAL_EXTRACTION",
    },
])

nontrivial_requirements_df

,required_quantity,purpose,status
0,Pi_mn[T],derive the physical source of Q_D,MISSING
1,alpha_t,derive temporal metric backreaction,MISSING
2,alpha_s,derive spatial metric backreaction,MISSING
3,second-order nonlinear coefficients,derive a2 and beta_GVH,MISSING
4,isotropic-coordinate transformation,separate gauge coefficients from PPN coefficients,REQUIRED_FOR_FINAL_EXTRACTION


Dans le prototype de `0.2.23`, la métrique faible est écrite sous une forme de travail :

\[
g_{tt}
=
-1+2u-2u^2+2\alpha_t d,
\]

\[
g_{\rm space}
=
1+2u+\frac32u^2+2\alpha_s d.
\]

Mais \(d(r)\) est Yukawa :

\[
d(r)=\frac{Q_D}{r}e^{-\mu_Dr}.
\]

Cette correction n’est pas en général un polynôme universel en \(u\). Les paramètres PPN peuvent alors être :

- dépendants de l’échelle ;
- dépendants de la source ;
- dépendants de la plage radiale de l’expérience.

Un ajustement numérique local ne constitue donc pas, à lui seul, une dérivation unique de \(a_1,a_2,b_1\).

In [12]:
alpha_t, alpha_s = sp.symbols(
    "alpha_t alpha_s",
    real=True,
)

Q_D, mu_D, r = sp.symbols(
    "Q_D mu_D r",
    positive=True,
    real=True,
)

d_yukawa = sp.simplify(
    Q_D
    * sp.exp(
        -mu_D * r
    )
    / r
)

nontrivial_metric_df = pd.DataFrame([
    {
        "metric_component": "g_tt",
        "expression": (
            "-1 + 2u - 2u^2 + 2 alpha_t d(r)"
        ),
        "coefficient_status": (
            "scale/source dependent unless reduced analytically"
        ),
    },
    {
        "metric_component": "g_space",
        "expression": (
            "1 + 2u + 3u^2/2 + 2 alpha_s d(r)"
        ),
        "coefficient_status": (
            "scale/source dependent unless reduced analytically"
        ),
    },
])

nontrivial_metric_df

,metric_component,expression,coefficient_status
0,g_tt,-1 + 2u - 2u^2 + 2 alpha_t d(r),scale/source dependent unless reduced analytically
1,g_space,1 + 2u + 3u^2/2 + 2 alpha_s d(r),scale/source dependent unless reduced analytically


# 8. Test d’extractibilité globale

In [13]:
unique_source_projector_derived = False
metric_backreaction_derived = False
nonlinear_beta_sector_derived = False
isotropic_gauge_fully_resolved = False

global_extraction_ready = bool(
    unique_source_projector_derived
    and metric_backreaction_derived
    and nonlinear_beta_sector_derived
    and isotropic_gauge_fully_resolved
)

global_readiness_df = pd.DataFrame([
    {
        "gate": "unique source projector",
        "pass": unique_source_projector_derived,
    },
    {
        "gate": "metric backreaction",
        "pass": metric_backreaction_derived,
    },
    {
        "gate": "nonlinear beta sector",
        "pass": nonlinear_beta_sector_derived,
    },
    {
        "gate": "isotropic gauge resolved",
        "pass": isotropic_gauge_fully_resolved,
    },
    {
        "gate": "global extraction ready",
        "pass": global_extraction_ready,
    },
])

global_readiness_df

,gate,pass
0,unique source projector,False
1,metric backreaction,False
2,nonlinear beta sector,False
3,isotropic gauge resolved,False
4,global extraction ready,False


# 9. Politique de promotion de l’artefact

In [14]:
PROMOTE_STRICT_BRANCH_AS_GLOBAL_MAPPING = False

if PROMOTE_STRICT_BRANCH_AS_GLOBAL_MAPPING:
    raise RuntimeError(
        "Promotion refusée par défaut : "
        "la branche stricte isotrope ne doit pas être "
        "présentée comme prédiction universelle de GVH."
    )

# 10. Artefact candidat — branche stricte

In [15]:
strict_branch_artifact = {
    "artifact_status": (
        "VALIDATED_FOR_RESTRICTED_BRANCH"
    ),
    "global_model_prediction": False,
    "branch_id": (
        "strict_isotropic_traceless_source"
    ),
    "scope": (
        "Perfect isotropic fluid with strict "
        "spatial traceless-stress coupling"
    ),
    "convention": "isotropic_ppn",
    "potential_definition": (
        "u = G_N M/(r c^2)"
    ),
    "metric_signature": "-+++",
    "g00": {
        "constant": "-1",
        "U_coefficient": "2",
        "U2_coefficient": "-2",
    },
    "gij": {
        "constant": "1",
        "U_coefficient": "2",
        "U2_coefficient": "3/2",
    },
    "normalized_coefficients": {
        "a1": "1",
        "a2": "1",
        "b1": "1",
    },
    "PPN": {
        "gamma_GVH": "1",
        "beta_GVH": "1",
    },
    "source_conditions": {
        "Pi_mn_T": "0",
        "Q_D": "0",
        "d_exterior": "0",
    },
    "gr_limit_substitutions": {},
    "source_notebooks": [
        "GVH_Diagonal_Cubic_0.2.21_"
        "Equivalence_Principle_PPN_Constraints.ipynb",
        "GVH_Diagonal_Cubic_0.2.22_"
        "Static_Spherical_Solution_PPN_Derivation.ipynb",
        "GVH_Diagonal_Cubic_0.2.23_"
        "Covariant_Static_Field_Equations_Source_Matching.ipynb",
    ],
    "warning": (
        "Do not use as a global GVH prediction. "
        "This artifact applies only to the restricted branch."
    ),
}

strict_branch_artifact

{'artifact_status': 'VALIDATED_FOR_RESTRICTED_BRANCH',
 'global_model_prediction': False,
 'branch_id': 'strict_isotropic_traceless_source',
 'scope': 'Perfect isotropic fluid with strict spatial traceless-stress coupling',
 'convention': 'isotropic_ppn',
 'potential_definition': 'u = G_N M/(r c^2)',
 'metric_signature': '-+++',
 'g00': {'constant': '-1', 'U_coefficient': '2', 'U2_coefficient': '-2'},
 'gij': {'constant': '1', 'U_coefficient': '2', 'U2_coefficient': '3/2'},
 'normalized_coefficients': {'a1': '1', 'a2': '1', 'b1': '1'},
 'PPN': {'gamma_GVH': '1', 'beta_GVH': '1'},
 'source_conditions': {'Pi_mn_T': '0', 'Q_D': '0', 'd_exterior': '0'},
 'gr_limit_substitutions': {},
 'source_notebooks': ['GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb',
  'GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb',
  'GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb'],
 'warning': 'Do not use as a global GVH prediction.

# 11. Artefact global bloqué

In [16]:
global_candidate_artifact = {
    "artifact_status": (
        "BLOCKED_NONTRIVIAL_COEFFICIENTS_NOT_DERIVED"
    ),
    "global_model_prediction": False,
    "convention": "isotropic_ppn",
    "potential_definition": (
        "u = G_N M/(r c^2)"
    ),
    "metric_signature": "-+++",
    "g00": {
        "constant": "-1",
        "U_coefficient": "TO_BE_DERIVED",
        "U2_coefficient": "TO_BE_DERIVED",
    },
    "gij": {
        "constant": "1",
        "U_coefficient": "TO_BE_DERIVED",
    },
    "parameters": {
        "alpha_t": "NOT_DERIVED",
        "alpha_s": "NOT_DERIVED",
        "Q_D": "SOURCE_DEPENDENT",
        "mu_D": "NOT_FIXED",
    },
    "gr_limit_substitutions": {
        "Q_D": "0"
    },
    "source_notebooks": [
        "GVH_Diagonal_Cubic_0.2.21_"
        "Equivalence_Principle_PPN_Constraints.ipynb",
        "GVH_Diagonal_Cubic_0.2.22_"
        "Static_Spherical_Solution_PPN_Derivation.ipynb",
        "GVH_Diagonal_Cubic_0.2.23_"
        "Covariant_Static_Field_Equations_Source_Matching.ipynb",
    ],
    "open_requirements": [
        "derive exact source projector Pi_mn[T]",
        "derive alpha_t and alpha_s from one fixed action",
        "derive second-order metric backreaction",
        "perform the final isotropic-coordinate expansion",
        "separate source-dependent Yukawa effects from standard PPN coefficients",
    ],
}

global_candidate_artifact

{'artifact_status': 'BLOCKED_NONTRIVIAL_COEFFICIENTS_NOT_DERIVED',
 'global_model_prediction': False,
 'convention': 'isotropic_ppn',
 'potential_definition': 'u = G_N M/(r c^2)',
 'metric_signature': '-+++',
 'g00': {'constant': '-1',
  'U_coefficient': 'TO_BE_DERIVED',
  'U2_coefficient': 'TO_BE_DERIVED'},
 'gij': {'constant': '1', 'U_coefficient': 'TO_BE_DERIVED'},
 'parameters': {'alpha_t': 'NOT_DERIVED',
  'alpha_s': 'NOT_DERIVED',
  'Q_D': 'SOURCE_DEPENDENT',
  'mu_D': 'NOT_FIXED'},
 'gr_limit_substitutions': {'Q_D': '0'},
 'source_notebooks': ['GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb',
  'GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb',
  'GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb'],
 'open_requirements': ['derive exact source projector Pi_mn[T]',
  'derive alpha_t and alpha_s from one fixed action',
  'derive second-order metric backreaction',
  'perform the final isotropic-coordinate

# 12. Décision scientifique

In [17]:
strict_branch_valid = bool(
    all_source_notebooks_found
    and strict_metric_pass
    and strict_ppn_pass
)

if not all_source_notebooks_found:
    FINAL_STATUS = (
        "BLOCKED-SOURCE-NOTEBOOKS-NOT-FOUND"
    )
elif global_extraction_ready:
    FINAL_STATUS = (
        "PASS-GLOBAL-GVH-WEAK-FIELD-COEFFICIENTS"
    )
elif strict_branch_valid:
    FINAL_STATUS = (
        "PASS-STRICT-ISOTROPIC-GR-BRANCH_"
        "BLOCKED-NONTRIVIAL-GVH-COEFFICIENTS-NOT-DERIVED"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-NONTRIVIAL-GVH-COEFFICIENTS-NOT-DERIVED"
    )

decision_df = pd.DataFrame([
    {
        "component": "source notebooks found",
        "pass": all_source_notebooks_found,
    },
    {
        "component": "strict isotropic branch",
        "pass": strict_branch_valid,
    },
    {
        "component": "nontrivial source projector",
        "pass": unique_source_projector_derived,
    },
    {
        "component": "metric backreaction",
        "pass": metric_backreaction_derived,
    },
    {
        "component": "nonlinear beta sector",
        "pass": nonlinear_beta_sector_derived,
    },
    {
        "component": "global GVH coefficient extraction",
        "pass": global_extraction_ready,
    },
])

print("STATUT FINAL :", FINAL_STATUS)
decision_df

STATUT FINAL : PASS-STRICT-ISOTROPIC-GR-BRANCH_BLOCKED-NONTRIVIAL-GVH-COEFFICIENTS-NOT-DERIVED


,component,pass
0,source notebooks found,True
1,strict isotropic branch,True
2,nontrivial source projector,False
3,metric backreaction,False
4,nonlinear beta sector,False
5,global GVH coefficient extraction,False


# 13. Exports

In [18]:
PREFIX = "GVH_Diagonal_Cubic_0.2.23.1"

exports = {
    "Source_Notebooks": source_notebook_df,
    "PPN_Interface_0.2.21": ppn_interface_021_df,
    "Parametric_Matching": matching_df,
    "Linear_Mapping": linear_mapping_df,
    "Source_Matching_Results": source_matching_results_df,
    "Strict_Branch": strict_branch_df,
    "Strict_Validation": strict_validation_df,
    "Nontrivial_Requirements": nontrivial_requirements_df,
    "Nontrivial_Metric": nontrivial_metric_df,
    "Global_Readiness": global_readiness_df,
    "Decision": decision_df,
}

for suffix, table in exports.items():
    table.to_csv(
        EXPORT_DIR
        / f"{PREFIX}_{suffix}.csv",
        index=False,
    )

STRICT_ARTIFACT_FILE = (
    PROCESSED_PPN_DIR
    / "gvh_weak_field_coefficients_"
    "STRICT_ISOTROPIC_BRANCH.json"
)

GLOBAL_CANDIDATE_FILE = (
    PROCESSED_PPN_DIR
    / "gvh_weak_field_coefficients_"
    "GLOBAL_CANDIDATE_BLOCKED.json"
)

STRICT_ARTIFACT_FILE.write_text(
    json.dumps(
        strict_branch_artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

GLOBAL_CANDIDATE_FILE.write_text(
    json.dumps(
        global_candidate_artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

metadata = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.2.23.1_"
        "Weak_Field_Coefficient_Extraction_for_PPN"
    ),
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "final_status": FINAL_STATUS,
    "strict_branch_valid": strict_branch_valid,
    "strict_branch_gamma": str(strict_gamma),
    "strict_branch_beta": str(strict_beta),
    "global_extraction_ready": global_extraction_ready,
    "canonical_global_artifact_written": False,
    "strict_branch_artifact": str(
        STRICT_ARTIFACT_FILE
    ),
    "blocked_global_candidate": str(
        GLOBAL_CANDIDATE_FILE
    ),
    "next_action": (
        "Derive Pi_mn[T], alpha_t, alpha_s and "
        "second-order metric backreaction from one "
        "fixed covariant action before creating the "
        "canonical gvh_weak_field_coefficients.json."
    ),
}

with (
    EXPORT_DIR
    / f"{PREFIX}_Metadata.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Strict artifact :", STRICT_ARTIFACT_FILE)
print("Blocked global candidate :", GLOBAL_CANDIDATE_FILE)
print("Canonical global artifact written : False")

Strict artifact : /content/Univers/gvh_diagonal_cubic/data/processed/ppn/gvh_weak_field_coefficients_STRICT_ISOTROPIC_BRANCH.json
Blocked global candidate : /content/Univers/gvh_diagonal_cubic/data/processed/ppn/gvh_weak_field_coefficients_GLOBAL_CANDIDATE_BLOCKED.json
Canonical global artifact written : False


# Conclusion

## Résultat dérivé

Pour une source parfaite isotrope et un couplage strict à la partie spatiale sans trace :

\[
\Pi_{\mu\nu}[T]=0,
\]

\[
Q_D=0,
\]

\[
d(r)=0,
\]

et l’extérieur est Schwarzschild.

On obtient alors :

\[
\boxed{
a_1=a_2=b_1=1
}
\]

et :

\[
\boxed{
\gamma_{\rm GVH}=1,
\qquad
\beta_{\rm GVH}=1
}
\]

dans cette branche précise.

## Résultat non encore dérivé

Pour une branche directionnelle non triviale :

\[
Q_D\neq0,
\]

les coefficients ne sont pas encore uniques, car il manque :

\[
\Pi_{\mu\nu}[T],
\qquad
\alpha_t,
\qquad
\alpha_s,
\]

et les termes non linéaires contrôlant \(a_2\).

Le notebook ne crée donc pas automatiquement :

```text
data/processed/ppn/gvh_weak_field_coefficients.json
```

car ce nom canonique serait interprété par `0.3.5` comme une prédiction globale prête à être validée.

Il produit à la place :

```text
gvh_weak_field_coefficients_STRICT_ISOTROPIC_BRANCH.json
```

et :

```text
gvh_weak_field_coefficients_GLOBAL_CANDIDATE_BLOCKED.json
```

Cette séparation empêche de transformer une limite GR restreinte en validation universelle de GVH.